In [37]:
"""
01_generate_portfolio.py
------------------------
Generate a synthetic retail/SME loan portfolio for PD scorecard development.

Two samples are produced:
  - Development sample (recent vintages, in-the-money)
  - Out-of-time (OOT) sample with mild distribution drift
    (bureau score and income) to exercise stability testing.

All data is synthetic - generated from a known data-generating process -
so model results can be compared against the "truth". 12-month default
horizon, point-in-time observation.

Run:  python 01_generate_portfolio.py
"""

import numpy as np
import pandas as pd
import os

rng = np.random.default_rng(42)

N_DEV = 25_000
N_OOT = 6_000

PRODUCTS = ["personal", "auto", "mortgage", "SME"]
EMPLOYMENT = ["salaried", "self_employed", "professional"]
REGIONS = ["metro", "tier1", "tier2", "tier3"]


def generate(n, oot=False):
    """Synthetic origination-level data. `oot=True` applies mild drift."""
    age = np.clip(rng.normal(38, 10, n), 21, 65).round(0)

    log_income = rng.normal(12.8, 0.45, n)
    if oot:  # income stress in later vintages
        log_income = log_income - 0.06
    annual_income = np.exp(log_income).round(-3)

    bureau_mean = 700 if not oot else 690  # bureau quality drifts down
    bureau_score = np.clip(rng.normal(bureau_mean, 80, n), 300, 900).round(0)

    product = rng.choice(PRODUCTS, n, p=[0.35, 0.20, 0.20, 0.25])
    employment = rng.choice(EMPLOYMENT, n, p=[0.55, 0.30, 0.15])
    region = rng.choice(REGIONS, n, p=[0.4, 0.25, 0.2, 0.15])

    # loan size scales with income; mortgages and SME are larger tickets
    size_mult = np.where(product == "mortgage", 4.0,
                np.where(product == "SME", 2.5, 1.0))
    loan_amount = (annual_income * size_mult * rng.uniform(0.5, 1.5, n)).round(-4)
    tenure_months = np.where(product == "mortgage", rng.integers(120, 300, n),
                      np.where(product == "auto", rng.integers(36, 84, n),
                      np.where(product == "SME", rng.integers(48, 120, n),
                               rng.integers(12, 60, n))))

    emi = loan_amount / tenure_months * 1.4  # rough annuity at ~14% p.a.
    other_emi = annual_income / 12 * rng.uniform(0.05, 0.35, n)
    dti_ratio = np.clip((emi + other_emi) / (annual_income / 12), 0, 3).round(3)

    delinq_24m = rng.poisson(0.25, n)
    months_since_delinq = np.where(delinq_24m > 0,
                                   rng.integers(1, 24, n), 99)

    # ---- latent default data-generating process (12-month PD) ----
    logit = (-3.90
             + 0.95 * (680 - bureau_score) / 80
             + 1.30 * (dti_ratio - 0.45)
             - 0.70 * (log_income - 12.8) / 0.45
             + 0.45 * (employment == "self_employed")
             + 0.40 * (product == "SME")
             - 0.30 * (product == "mortgage")
             + 0.40 * np.minimum(delinq_24m, 3)
             + 0.25 * (region == "tier3"))
    pd_true = 1 / (1 + np.exp(-logit))
    default_12m = rng.binomial(1, pd_true)

    df = pd.DataFrame({
        "age": age,
        "annual_income": annual_income,
        "bureau_score": bureau_score,
        "product_type": product,
        "employment_type": employment,
        "region": region,
        "loan_amount": loan_amount,
        "tenure_months": tenure_months,
        "dti_ratio": dti_ratio,
        "delinq_24m": delinq_24m,
        "months_since_delinq": months_since_delinq,
        "pd_true": pd_true.round(5),
        "default_12m": default_12m,
        # outstanding balance as EAD proxy (70-100% of origination)
        "ead": (loan_amount * rng.uniform(0.70, 1.0, n)).round(-3),
        "sample": "oot" if oot else "dev",
    })
    return df


dev = generate(N_DEV, oot=False)
oot = generate(N_OOT, oot=True)

os.makedirs("data", exist_ok=True) # Create the directory if it doesn't exist
dev.to_csv("data/portfolio_dev.csv", index=False)
oot.to_csv("data/portfolio_oot.csv", index=False)

print(f"Development sample : {len(dev):,} rows | default rate {dev['default_12m'].mean():.2%}")
print(f"Out-of-time sample : {len(oot):,} rows | default rate {oot['default_12m'].mean():.2%}")
print("\nSegment default rates (dev):")
print(dev.groupby("product_type")["default_12m"].agg(["count", "mean"]).round(4))

Development sample : 25,000 rows | default rate 5.49%
Out-of-time sample : 6,000 rows | default rate 7.02%

Segment default rates (dev):
              count    mean
product_type               
SME            6248  0.0775
auto           4987  0.0419
mortgage       5047  0.0321
personal       8718  0.0594


In [38]:
"""
02_woe_iv_segmentation.py
-------------------------
Exploratory risk analysis, univariate WOE/IV screening, and the
segmentation assessment for the PD scorecard.

Outputs:
  outputs/woe_iv_summary.csv   - IV per candidate feature
  outputs/woe_bureau_score.csv - WOE table for the strongest feature
  figures/segment_default_rates.png
  figures/woe_bureau_score.png

Run:  python 02_woe_iv_segmentation.py
"""

import sys
import os

# Add the current directory to sys.path to allow importing local modules
# This assumes scorecard_lib.py is in the same directory as this notebook.
if '.' not in sys.path:
    sys.path.insert(0, '.')

print(f"Current working directory added to sys.path: {os.getcwd()}")

Current working directory added to sys.path: /content


In [41]:
%%writefile scorecard_lib.py
# === BEGIN scorecard_lib.py CONTENT ===
import pandas as pd
import numpy as np

def fit_woe(X, y, categorical=True, max_bins=5, min_samples_leaf=0.05, min_samples_bin=0.03):
    # Ensure X is a Series and y is a Series
    if not isinstance(X, pd.Series): X = pd.Series(X)
    if not isinstance(y, pd.Series): y = pd.Series(y)

    df = pd.DataFrame({'feature': X, 'target': y}).copy()
    df['target'] = df['target'].astype(int)

    # Handle missing values by treating them as a separate category/bin
    if df['feature'].isnull().any():
        df['feature'] = df['feature'].fillna('Missing')

    total_good = (df['target'] == 0).sum()
    total_bad = (df['target'] == 1).sum()

    if total_good == 0 or total_bad == 0:
        # Cannot calculate WOE if no good or no bad observations
        return {"iv": 0.0, "table": pd.DataFrame(
            columns=['bin', 'count', 'good', 'bad', 'total_target_0', 'total_target_1', 'prop_good', 'prop_bad', 'woe', 'iv'])}

    if categorical:
        group_by_col = 'feature'
        bin_df = df.groupby(group_by_col, observed=True).agg(
            count=('feature', 'size'),
            good=('target', lambda x: (x == 0).sum()),
            bad=('target', lambda x: (x == 1).sum())
        ).reset_index()
        bin_df = bin_df.rename(columns={'feature': 'bin'})

    else: # Numeric feature
        # Use qcut for binning, handling cases with few unique values or too many bins
        try:
            bins = pd.qcut(df['feature'], q=max_bins, duplicates='drop', retbins=True)[1]
            if len(bins) == 1: # If only one unique value or cannot create multiple bins
                bins = pd.cut(df['feature'], bins=max_bins, duplicates='drop', retbins=True)[1]

            # Ensure bins are sorted and unique
            bins = np.unique(bins)

            # If there's only one bin, it means all values are the same or qcut/cut failed to create multiple.
            # In this case, treat as a single bin or return 0 IV.
            if len(bins) <= 1: # Handle cases where qcut might return only 1 bin edge (e.g., all values are same)
                bin_labels = [f"[{df['feature'].min()}, {df['feature'].max()}]"]
                df['binned_feature'] = bin_labels[0]
            else:
                df['binned_feature'] = pd.cut(df['feature'], bins=bins, include_lowest=True, precision=4, duplicates='drop')
                bin_labels = [str(x) for x in df['binned_feature'].unique() if pd.notna(x)]

            bin_df = df.groupby('binned_feature', observed=True).agg(
                count=('feature', 'size'),
                good=('target', lambda x: (x == 0).sum()),
                bad=('target', lambda x: (x == 1).sum())
            ).reset_index()
            bin_df = bin_df.rename(columns={'binned_feature': 'bin'})

        except Exception as e:
            # If binning fails, treat as a single bin or return 0 IV
            print(f"Warning: Binning failed for numeric feature. Treating as single bin. Error: {e}")
            bin_labels = [f"[{df['feature'].min()}, {df['feature'].max()}]"]
            bin_df = df.groupby(pd.Series(bin_labels[0], index=df.index), observed=True).agg(
                count=('feature', 'size'),
                good=('target', lambda x: (x == 0).sum()),
                bad=('target', lambda x: (x == 1).sum())
            ).reset_index()
            bin_df = bin_df.rename(columns={'index': 'bin'})


    # Add a small constant to avoid division by zero or log(0)
    epsilon = 0.000001

    bin_df['total_target_0'] = total_good
    bin_df['total_target_1'] = total_bad

    bin_df['prop_good'] = (bin_df['good'] + epsilon) / (total_good + epsilon)
    bin_df['prop_bad'] = (bin_df['bad'] + epsilon) / (total_bad + epsilon)

    bin_df['woe'] = np.log(bin_df['prop_good'] / bin_df['prop_bad'])
    bin_df['iv'] = (bin_df['prop_good'] - bin_df['prop_bad']) * bin_df['woe']

    total_iv = bin_df['iv'].sum()

    # Sort by WOE for better interpretability, especially for numeric bins
    if not categorical:
        bin_df['sort_key'] = bin_df['bin'].apply(lambda x: x.left if hasattr(x, 'left') else float('-inf'))
        bin_df = bin_df.sort_values(by='sort_key').drop(columns='sort_key')

    # Reorder columns for consistency
    final_columns = ['bin', 'count', 'good', 'bad', 'woe', 'iv']
    result_table = bin_df[final_columns]

    return {"iv": total_iv, "table": result_table}

def iv_strength(iv_value):
    if iv_value < 0.02:
        return "Weak"
    elif 0.02 <= iv_value < 0.1:
        return "Medium"
    elif 0.1 <= iv_value < 0.3:
        return "Strong"
    elif 0.3 <= iv_value < 0.5:
        return "Very Strong"
    else:
        return "Suspiciously Strong"

# === END scorecard_lib.py CONTENT ===

Overwriting scorecard_lib.py


In [40]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import importlib # Added importlib
import os # Import os module to create directories

import scorecard_lib # Changed to import the module directly
importlib.reload(scorecard_lib) # Reload the module
from scorecard_lib import fit_woe, iv_strength # Then import functions from the reloaded module

dev = pd.read_csv("data/portfolio_dev.csv")
oot = pd.read_csv("data/portfolio_oot.csv") # Load the oot dataset

NUMERIC = ["age", "annual_income", "bureau_score", "loan_amount",
           "tenure_months", "dti_ratio", "delinq_24m", "months_since_delinq"]
CATEGORICAL = ["product_type", "employment_type", "region"]

TARGET = "default_12m"

# Create 'outputs' and 'figures' directories if they don't exist
os.makedirs("outputs", exist_ok=True)
os.makedirs("figures", exist_ok=True)

# ---------- 1. Univariate WOE / IV screening ----------
rows = []
for col in NUMERIC:
    spec = fit_woe(dev[col], dev[TARGET], categorical=False, max_bins=10)
    rows.append({"feature": col, "type": "numeric", "iv": round(spec["iv"], 4),
                 "strength": iv_strength(spec["iv"])})
for col in CATEGORICAL:
    spec = fit_woe(dev[col], dev[TARGET], categorical=True)
    rows.append({"feature": col, "type": "categorical", "iv": round(spec["iv"], 4),
                 "strength": iv_strength(spec["iv"])})

iv_summary = pd.DataFrame(rows).sort_values("iv", ascending=False)
iv_summary.to_csv("outputs/woe_iv_summary.csv", index=False)
print("Information Value summary (12-month default):")
print(iv_summary.to_string(index=False))

# ---------- 2. WOE table for the strongest numeric driver ----------
bureau_spec = fit_woe(dev["bureau_score"], dev[TARGET], categorical=False)
tab = bureau_spec["table"][["bin", "count", "good", "bad", "woe", "iv"]]
tab.to_csv("outputs/woe_bureau_score.csv", index=False)
print("\nWOE - bureau_score:")
print(tab.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
# Convert Interval objects to strings before processing for labels
labels = [str(l).split(",")[0].strip("(") for l in tab["bin"]]
ax.bar(range(len(tab)), tab["woe"], color="#1F3864")
ax.set_xticks(range(len(tab)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Bureau score (bin lower edge)")
ax.set_ylabel("WOE")
ax.set_title("Weight of Evidence by bureau score bin")
fig.tight_layout()
fig.savefig("figures/woe_bureau_score.png", dpi=150)

# ---------- 3. Segmentation assessment ----------
seg_dev = dev.groupby("product_type").agg(
    n=(TARGET, "size"),
    default_rate=(TARGET, "mean"),
    avg_ead=("ead", "mean"),
).round(4)
print("\nSegment profile (Development Sample):")
print(seg_dev.to_string())

seg_oot = oot.groupby("product_type").agg(
    n=(TARGET, "size"),
    default_rate=(TARGET, "mean"),
    avg_ead=("ead", "mean"),
).round(4)
print("\nSegment profile (Out-of-Time Sample):")
print(seg_oot.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(seg_dev.index, seg_dev["default_rate"], color="#1F3864")
ax.set_ylabel("12-month default rate")
ax.set_title("Default rate by product segment (development sample)")
for i, (idx, r) in enumerate(seg_dev.iterrows()):
    ax.text(i, r["default_rate"] + 0.001, f"{r['default_rate']:.2%}\nn={int(r['n']):,}",
            ha="center", fontsize=9)
fig.tight_layout()
fig.savefig("figures/segment_default_rates.png", dpi=150)

# Monotonic risk ordering check on bureau score
print("\nBinned default rate vs bureau score (monotonicity check):")
brk = [-np.inf, 480, 540, 600, 660, 720, 780, np.inf]
check = dev.groupby(pd.cut(dev["bureau_score"], brk), observed=False)[TARGET].agg(["count", "mean"])
print(check.round(4).to_string())

print("\nNotes for segmentation decision:")
print("- SME default rate materially above other products; mortgage below.")
print("- Product type will enter the pooled model as a WOE-encoded feature;")
print("  a dedicated segmented-vs-pooled performance comparison is run in")
print("  03_pd_scorecard_development.py before finalising the structure.")


Information Value summary (12-month default):
            feature        type     iv            strength
       bureau_score     numeric 0.7431 Suspiciously Strong
      annual_income     numeric 0.3851         Very Strong
          dti_ratio     numeric 0.1391              Strong
        loan_amount     numeric 0.1196              Strong
       product_type categorical 0.1065              Strong
      tenure_months     numeric 0.0887              Medium
    employment_type categorical 0.0525              Medium
         delinq_24m     numeric 0.0272              Medium
months_since_delinq     numeric 0.0267              Medium
             region categorical 0.0171                Weak
                age     numeric 0.0102                Weak

WOE - bureau_score:
              bin  count  good  bad       woe       iv
(386.9999, 633.0]   5059  4369  690 -0.999794 0.317568
   (633.0, 680.0]   5043  4729  314 -0.133316 0.003805
   (680.0, 720.0]   4997  4800  197  0.347775 0.020754
   (7